# Q1: vitamin D and incident CVD

**Question:** How is baseline measured vitamin D associated with subsequent incident MI/stroke?

**Inputs:** Prepared Q1 cohort with follow-up, events and baseline covariates.

**Outputs:** Categorical Cox estimates, BMI sensitivity, ordinal trend, continuous reference HR, subgroup and interaction tables.

The primary categorical model and its original companion BMI/trend/continuous checks are labeled separately in the output. This source retains the existing complete-case rules and formulas. Forest plotting is separated into the presentation stage.


## Configuration and cohort


In [ ]:
from pathlib import Path
import os
import sys

if os.environ.get("CAPSTONE_PROJECT_ROOT"):
    PROJECT_ROOT = Path(os.environ["CAPSTONE_PROJECT_ROOT"]).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "shared" / "q1q2_common.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the example folder, or set CAPSTONE_PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT / "shared"))
from q1q2_common import *

base = load_q1_analysis_base()


## Cox model definitions


In [ ]:
def lifelines_rows(model: CoxPHFitter, label: str, terms: list[str]) -> pd.DataFrame:
    rows = []
    for term in terms:
        if term not in model.summary.index:
            continue
        row = model.summary.loc[term]
        rows.append(
            {
                "model": label,
                "term": term,
                "n": int(model._n_examples),
                "events": int(model.event_observed.sum()),
                "hr": row["exp(coef)"],
                "lo": row["exp(coef) lower 95%"],
                "hi": row["exp(coef) upper 95%"],
                "p": row["p"],
            }
        )
    return pd.DataFrame(rows)


def run_q1_main_cox(base: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    d = base.copy()
    d["vitd_deficient"] = (d["vitd_main"] < 10).astype(int)
    d["vitd_insufficient"] = ((d["vitd_main"] >= 10) & (d["vitd_main"] < 20)).astype(int)
    d["vitd_adequate"] = ((d["vitd_main"] >= 20) & (d["vitd_main"] < 30)).astype(int)
    d["vitd_trend"] = d["vitd_main"].apply(vitamin_d_trend)
    d["vitd_neg10"] = -d["vitd_main"] / 10.0

    exposure_terms = ["vitd_deficient", "vitd_insufficient", "vitd_adequate"]
    base_cols = ["follow_up_years", "event_mi_or_stroke"] + exposure_terms + Q1_COVARS
    model_a_df = d[base_cols].dropna()
    model_a = CoxPHFitter().fit(model_a_df, "follow_up_years", "event_mi_or_stroke")

    tables = [lifelines_rows(model_a, "categorical_no_bmi", exposure_terms)]

    if "bmi_full" in d.columns:
        bmi_df = d[(d["bmi_full"] >= 12) & (d["bmi_full"] <= 70)].copy()
        model_b_df = bmi_df[base_cols + ["bmi_full"]].dropna()
        model_b = CoxPHFitter().fit(model_b_df, "follow_up_years", "event_mi_or_stroke")
        tables.append(lifelines_rows(model_b, "categorical_plus_bmi", exposure_terms + ["bmi_full"]))

    trend_cols = ["follow_up_years", "event_mi_or_stroke", "vitd_trend"] + Q1_COVARS
    trend_df = d[trend_cols].dropna()
    trend_model = CoxPHFitter().fit(trend_df, "follow_up_years", "event_mi_or_stroke")
    trend_table = lifelines_rows(trend_model, "ordinal_trend", ["vitd_trend"])
    trend_table.to_csv(table_path("q1_trend_test.csv"), index=False)

    cont_cols = ["follow_up_years", "event_mi_or_stroke", "vitd_neg10"] + Q1_COVARS
    cont_df = d[cont_cols].dropna()
    cont_model = CoxPHFitter().fit(cont_df, "follow_up_years", "event_mi_or_stroke")
    overall = lifelines_rows(cont_model, "continuous_per_10_ng_ml_lower", ["vitd_neg10"])
    overall.rename(columns={"hr": "hr", "lo": "lo", "hi": "hi", "p": "p"}, inplace=True)
    overall[["hr", "lo", "hi", "p", "n", "events"]].to_csv(table_path("q1_overall_hr.csv"), index=False)

    cox_table = pd.concat(tables + [trend_table, overall], ignore_index=True)
    cox_table.to_csv(table_path("q1_cox_models_clean.csv"), index=False)

    print_section("Q1 Cox models")
    print(cox_table.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    print("Saved q1_cox_models_clean.csv, q1_trend_test.csv, q1_overall_hr.csv")
    return cox_table, overall


## Fit the Cox models


In [ ]:
cox_table, overall = run_q1_main_cox(base)


## Subgroups and interaction tests


In [ ]:
def add_income_band_if_available(df: pd.DataFrame) -> pd.DataFrame:
    try:
        qc = run_query(
            f"""
            SELECT o.observation_concept_id AS cid,
                   c.concept_name AS name,
                   COUNT(DISTINCT o.person_id) AS n
            FROM `{CDR}.observation` o
            JOIN `{CDR}.concept` c
              ON o.observation_concept_id = c.concept_id
            WHERE LOWER(c.concept_name) LIKE '%income%'
            GROUP BY 1, 2
            ORDER BY n DESC
            LIMIT 1
            """
        )
        if qc.empty:
            return df
        income_cid = int(qc.iloc[0]["cid"])
        inc = run_query(
            f"""
            SELECT o.person_id, c.concept_name AS ans
            FROM `{CDR}.observation` o
            JOIN `{CDR}.concept` c
              ON o.value_as_concept_id = c.concept_id
            WHERE o.observation_concept_id = {income_cid}
            """
        )
        inc = as_person_id(inc)

        def band(answer: object) -> str | None:
            text = str(answer).lower()
            if any(k in text for k in ["less 10", "10k 25", "25k 35", "less than 10"]):
                return "Income <35k"
            if any(k in text for k in ["35k 50", "50k 75", "75k 100"]):
                return "Income 35-100k"
            if any(k in text for k in ["100k 150", "150k 200", "more 200"]):
                return "Income >=100k"
            return None

        inc["g_income"] = inc["ans"].apply(band)
        inc = inc.dropna(subset=["g_income"]).drop_duplicates("person_id")
        out = df.merge(inc[["person_id", "g_income"]], on="person_id", how="left")
        print(f"Income coverage in Q1 base: {out['g_income'].notna().sum():,}")
        return out
    except Exception as exc:
        print(f"Income subgroup skipped: {exc}")
        return df

def run_q1_subgroups(base: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    d = base.copy()
    d["vitd_neg10"] = -d["vitd_main"] / 10.0
    d["g_age"] = np.where(d["age_at_index"] >= 65, "Age >=65", "Age <65")
    d["g_sex"] = np.where(d["female"] == 1, "Female", "Male")
    d["g_diabetes"] = np.where(d["baseline_diabetes"] == 1, "Yes", "No")
    d["g_hypertension"] = np.where(d["baseline_hypertension"] == 1, "Yes", "No")
    d["g_hyperlipidemia"] = np.where(d["baseline_hyperlipidemia"] == 1, "Yes", "No")
    if "race_eth_simplified" in d.columns:
        d["g_race"] = d["race_eth_simplified"]
    if "bmi_full" in d.columns:
        d["g_bmi"] = pd.cut(
            d["bmi_full"],
            [0, 25, 30, 200],
            labels=["BMI <25", "BMI 25-30", "BMI >=30"],
        ).astype(object)
    d = add_income_band_if_available(d)

    exclude = {
        "g_age": ["age_at_index"],
        "g_sex": ["female"],
        "g_race": ["race_black", "race_hispanic", "race_asian", "race_other"],
        "g_diabetes": ["baseline_diabetes"],
        "g_hypertension": ["baseline_hypertension"],
        "g_hyperlipidemia": ["baseline_hyperlipidemia"],
        "g_bmi": [],
        "g_income": [],
    }

    specs = [
        ("g_age", "Age"),
        ("g_sex", "Sex"),
        ("g_race", "Race/ethnicity"),
        ("g_bmi", "BMI"),
        ("g_diabetes", "Diabetes"),
        ("g_hypertension", "Hypertension"),
        ("g_hyperlipidemia", "Hyperlipidemia"),
        ("g_income", "Household income"),
    ]
    specs = [(col, label) for col, label in specs if col in d.columns]

    def covars_for(group_col: str) -> list[str]:
        return [c for c in Q1_COVARS if c not in exclude[group_col]]

    def fit_hr(sub: pd.DataFrame, adj: list[str]) -> tuple[float, float, float, float]:
        cols = ["follow_up_years", "event_mi_or_stroke", "vitd_neg10"] + adj
        m = sub[cols].dropna()
        m = m.loc[:, m.nunique() > 1]
        model = CoxPHFitter().fit(m, "follow_up_years", "event_mi_or_stroke")
        row = model.summary.loc["vitd_neg10"]
        return row["exp(coef)"], row["exp(coef) lower 95%"], row["exp(coef) upper 95%"], row["p"]

    def interaction_p(group_col: str) -> float:
        dd = d.dropna(subset=[group_col]).copy()
        dum = pd.get_dummies(dd[group_col].astype(str), prefix="g", drop_first=True).astype(float)
        dd = pd.concat([dd.reset_index(drop=True), dum.reset_index(drop=True)], axis=1)
        dum_cols = list(dum.columns)
        interaction_cols = []
        for dum_col in dum_cols:
            col = "ix_" + dum_col
            dd[col] = dd["vitd_neg10"] * dd[dum_col]
            interaction_cols.append(col)
        adj = covars_for(group_col)
        cols = ["follow_up_years", "event_mi_or_stroke", "vitd_neg10"] + adj + dum_cols
        reduced = dd[cols].dropna()
        full = dd[cols + interaction_cols].dropna()
        cr = CoxPHFitter(penalizer=0.0).fit(reduced, "follow_up_years", "event_mi_or_stroke")
        cf = CoxPHFitter(penalizer=0.0).fit(full, "follow_up_years", "event_mi_or_stroke")
        lr = 2 * (cf.log_likelihood_ - cr.log_likelihood_)
        return stats.chi2.sf(lr, len(interaction_cols))

    rows = []
    interactions = []
    for group_col, label in specs:
        for level in sorted(d[group_col].dropna().unique()):
            sub = d[d[group_col] == level]
            events = int(sub["event_mi_or_stroke"].sum())
            if len(sub) < 300 or events < 25:
                continue
            try:
                hr, lo, hi, p = fit_hr(sub, covars_for(group_col))
                rows.append(
                    {
                        "subgroup": label,
                        "level": str(level),
                        "n": len(sub),
                        "events": events,
                        "hr": hr,
                        "lo": lo,
                        "hi": hi,
                        "p": p,
                    }
                )
            except Exception as exc:
                print(f"Subgroup failed: {label}={level}: {exc}")
        try:
            interactions.append({"subgroup": label, "p_interaction": interaction_p(group_col)})
        except Exception as exc:
            print(f"Interaction failed: {label}: {exc}")
            interactions.append({"subgroup": label, "p_interaction": np.nan})

    res = pd.DataFrame(rows)
    itab = pd.DataFrame(interactions)
    res.to_csv(table_path("q1_subgroups_all.csv"), index=False)
    itab.to_csv(table_path("q1_interaction_tests.csv"), index=False)
    if not itab.empty:
        bonf = itab.copy()
        bonf["n_tests"] = len(bonf)
        bonf["bonferroni_threshold"] = 0.05 / len(bonf)
        bonf["bonferroni_pass"] = bonf["p_interaction"] < bonf["bonferroni_threshold"]
        bonf.to_csv(table_path("q1_interaction_bonferroni.csv"), index=False)
    print_section("Q1 subgroup analysis")
    print(res.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    print("\nInteraction tests")
    print(itab.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    if not itab.empty:
        print("\nBonferroni correction")
        print(bonf.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    print("Saved q1_subgroups_all.csv, q1_interaction_tests.csv, and q1_interaction_bonferroni.csv")
    return res, itab


## Fit subgroup models


In [ ]:
subgroups, interactions = run_q1_subgroups(base)
